# 第 5 周练习：Drive Sage —— 本地检索你的 Google Drive 知识库

## 练习目标（理念）

在**完全本地**处理的前提下（嵌入与聊天走 Ollama），从 Google Drive 同步文档 → 分块入库 → RAG 问答。也可上传本地文件即时索引。

## 和第 5 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 文档加载与分块 | Drive 下载 / docx·md·txt；Markdown 按标题再切 |
| 向量库 | Chroma + `OllamaEmbeddings`（如 nomic-embed-text） |
| RAG | `similarity_search_with_score` + 距离阈值 + 本地 `ChatOllama` |
| 工程细节 | OAuth、manifest 缓存、增量同步 |

## 怎么跑

1. 本机 Ollama 已拉取嵌入模型与聊天模型（可用环境变量覆盖默认名）
2. 把 Google Cloud 的 `client_secret_....json` 放在笔记本同目录
3. 运行依赖安装格 → 授权 Drive → Sync → 提问


In [ ]:
# 安装本练习依赖（版本钉死一段，减少 LangChain 1.x 破坏性变更）
%pip install -qU "langchain==0.3.27" "langchain-core<1.0.0,>=0.3.78" "langchain-text-splitters<1.0.0,>=0.3.9" langchain_ollama langchain_chroma langchain_community google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client python-docx



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ========== 导入：标准库 / Gradio / LangChain / Google Drive / docx ==========

# 导入 json：读写 manifest 缓存
import json
# 导入 logging：同步/下载过程可观测
import logging
# 导入 os：读环境变量（距离阈值、模型名）
import os
# 导入 re：清洗文件名等
import re
# 导入 sys：日志打到 stdout
import sys
# 导入 hashlib：本地文件内容指纹（增量索引）
import hashlib
# 从 pathlib 导入 Path：统一路径与缓存目录
from pathlib import Path
# 从 enum 导入 StrEnum：元数据字段名枚举，避免魔法字符串
from enum import StrEnum
# 类型标注：Iterable / Optional
from typing import Iterable, Optional

# Gradio：聊天 + Drive 同步 UI
import gradio as gr
# LangChain Document：page_content + metadata
from langchain_core.documents import Document
# 切块器：通用递归切分 + Markdown 按标题切
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
# 本地 Ollama：嵌入与聊天
from langchain_ollama import OllamaEmbeddings, ChatOllama
# 父文档内存仓：检索到 chunk 后可回看 parent
from langchain.storage import InMemoryStore
# Chroma 向量库封装
from langchain_chroma import Chroma
# 纯文本/Markdown 加载器
from langchain_community.document_loaders import TextLoader
# Google OAuth 凭证对象
from google.oauth2.credentials import Credentials
# 本地浏览器 OAuth 流程
from google_auth_oauthlib.flow import InstalledAppFlow
# 刷新过期 token 时用的 HTTP Request
from google.auth.transport.requests import Request
# 构建 Drive API client
from googleapiclient.discovery import build
# 流式下载 Drive 文件
from googleapiclient.http import MediaIoBaseDownload
# Drive API 错误类型
from googleapiclient.errors import HttpError
# python-docx：解析 .docx（别名避免与 LangChain Document 冲突）
from docx import Document as DocxDocument


In [ ]:
# ========== 日志：drive_sage logger 打到 stdout ==========

# 专用 logger 名，方便过滤
logger = logging.getLogger('drive_sage')
# DEBUG：同步进度、距离过滤等细节可见
logger.setLevel(logging.DEBUG)

# 避免 Jupyter 重复跑单元格时叠加多个 handler
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)


In [ ]:
# ========== 路径 / 文件类型映射 / 模型常量 / Gradio CSS ==========

# OAuth 范围：只读 Drive（最小权限）
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
# 笔记本当前工作目录；client_secret 与缓存都相对这里
APP_ROOT = Path.cwd()
# 本地数据根目录（下载、向量库、token、manifest）
DATA_DIR = APP_ROOT / '.drive_sage'
DOWNLOAD_DIR = DATA_DIR / 'downloads'
VECTORSTORE_DIR = DATA_DIR / 'chroma'
TOKEN_PATH = DATA_DIR / 'token.json'
MANIFEST_PATH = DATA_DIR / 'manifest.json'
CLIENT_SECRET_FILE = APP_ROOT / 'client_secret_202216035337-4qson0c08g71u8uuihv6v46arv64nhvg.apps.googleusercontent.com.json'

# 确保缓存目录存在
for path in (DATA_DIR, DOWNLOAD_DIR, VECTORSTORE_DIR):
    path.mkdir(parents=True, exist_ok=True)

# UI 可选文件类型：label / 扩展名 / Drive MIME
FILE_TYPE_OPTIONS = {
    'txt': {
        'label': '.txt - Plain text',
        'extensions': ['.txt'],
        'mime_types': ['text/plain'],
    },
    'md': {
        'label': '.md - Markdown',
        'extensions': ['.md'],
        'mime_types': ['text/markdown', 'text/plain'],
    },
    'docx': {
        'label': '.docx - Word (OpenXML)',
        'extensions': ['.docx'],
        'mime_types': ['application/vnd.openxmlformats-officedocument.wordprocessingml.document'],
    },
    'doc': {
        'label': '.doc - Word 97-2003',
        'extensions': ['.doc'],
        'mime_types': ['application/msword', 'application/vnd.ms-word.document.macroenabled.12'],
    },
    'gdoc': {
        'label': 'Google Docs (exported)',
        'extensions': ['.docx'],
        'mime_types': ['application/vnd.google-apps.document'],
    },
}

# label → key，方便 CheckboxGroup 回传
FILE_TYPE_LABEL_TO_KEY = {config['label']: key for key, config in FILE_TYPE_OPTIONS.items()}
DEFAULT_FILE_TYPE_KEYS = ['txt', 'md', 'docx', 'doc', 'gdoc']
DEFAULT_FILE_TYPE_LABELS = [FILE_TYPE_OPTIONS[key]['label'] for key in DEFAULT_FILE_TYPE_KEYS]

# MIME → 默认扩展名（下载落盘用）
MIME_TYPE_TO_EXTENSION = {}
for key, config in FILE_TYPE_OPTIONS.items():
    extension = config['extensions'][0]
    for mime in config['mime_types']:
        MIME_TYPE_TO_EXTENSION[mime] = extension

# Google Docs 导出为 docx 再解析
GOOGLE_EXPORT_FORMATS = {
    'application/vnd.google-apps.document': (
        'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
        '.docx'
    ),
}

# 相似度距离上限：越大越松（可用环境变量覆盖）
SIMILARITY_DISTANCE_MAX = float(os.getenv('DRIVE_SAGE_DISTANCE_MAX', '1.2'))
# 拼进 prompt 的单段上下文最大字符数
MAX_CONTEXT_SNIPPET_CHARS = 1200
# 读文件计算哈希时的块大小
HASH_BLOCK_SIZE = 65536
# 本地嵌入模型名（Ollama）
EMBED_MODEL = os.getenv('DRIVE_SAGE_EMBED_MODEL', 'nomic-embed-text')
# 本地聊天模型名（Ollama）
CHAT_MODEL = os.getenv('DRIVE_SAGE_CHAT_MODEL', 'llama3.1:latest')

# Gradio CSS：用 elem_id 控制聊天列高度（选择器必须是 #id，不能夹杂注释文字）
CUSTOM_CSS = """
#chat-column {
    height: 80vh;
}
#chat-column > div {
    height: 100%;
}
#chat-column .gradio-chatbot,
#chat-column .gradio-chat-interface,
#chat-column .gradio-chatinterface {
    height: 100%;
}
#chat-output {
    height: 100%;
}
#chat-output .overflow-y-auto {
    max-height: 100% !important;
}
#chat-output .h-full {
    height: 100% !important;
}
"""


In [ ]:
# ========== Google Drive 授权：复用 token / 刷新 / 本地 OAuth ==========

def build_drive_service():
    creds = None
    # 优先读本地缓存的 token.json
    if TOKEN_PATH.exists():
        try:
            creds = Credentials.from_authorized_user_file(str(TOKEN_PATH), SCOPES)
        except Exception as exc:
            logger.warning('Failed to load cached credentials: %s', exc)
            # 坏掉的 token 删掉，迫使重新登录
            TOKEN_PATH.unlink(missing_ok=True)
            creds = None

    if not creds or not creds.valid:
        # 过期但有 refresh_token：静默刷新
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())
            except Exception as exc:
                logger.warning('Refreshing credentials failed: %s', exc)
                creds = None

        if not creds or not creds.valid:
            # 需要交互登录：先确认 client_secret 文件在
            if not CLIENT_SECRET_FILE.exists():
                raise FileNotFoundError(
                    'client_secret.json not found. Download it from Google Cloud Console and place it next to this notebook.'
                )
            flow = InstalledAppFlow.from_client_secrets_file(str(CLIENT_SECRET_FILE), SCOPES)
            # port=0：系统分配空闲端口打开本地回调
            creds = flow.run_local_server(port=0)

        # 持久化，下次免登录
        with TOKEN_PATH.open('w', encoding='utf-8') as token_file:
            token_file.write(creds.to_json())
            
    return build('drive', 'v3', credentials=creds)


In [ ]:
# ========== manifest：记录已同步文件的 modified / 本地路径 ==========

def load_manifest() -> dict:
    if MANIFEST_PATH.exists():
        try:
            with MANIFEST_PATH.open('r', encoding='utf-8') as fp:
                raw = json.load(fp)
            # 兼容旧格式：值可能只是 modified 字符串
            if isinstance(raw, dict):
                normalized: dict[str, dict] = {}
                for file_id, entry in raw.items():
                    if isinstance(entry, dict):
                        normalized[file_id] = entry
                    else:
                        normalized[file_id] = {'modified': str(entry)}
                return normalized
        except json.JSONDecodeError:
            logger.warning('Manifest file is corrupted; resetting cache.')
    return {}

# 写回磁盘（缩进 JSON，便于人工查看）
def save_manifest(manifest: dict) -> None:
    with MANIFEST_PATH.open('w', encoding='utf-8') as fp:
        json.dump(manifest, fp, indent=2)


In [ ]:
# ========== 元数据字段 + 向量库 / 嵌入 / 切块器初始化 ==========

# 统一 metadata 键名，避免拼写漂移
class Metadata(StrEnum):
    ID = 'id'
    SOURCE = 'source'
    PARENT_ID = 'parent_id'
    FILE_TYPE = 'file_type'
    TITLE = 'title'
    MODIFIED = 'modified'

def metadata_key(key: Metadata) -> str:
    return key.value

# 本地嵌入：需 Ollama 已 pull 对应模型
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

try:
    # 内存/默认客户端上的 collection（本练习强调本地会话）
    vectorstore = Chroma(
        collection_name='drive_sage',
        embedding_function=embeddings,
    )
except Exception as exc:
    logger.exception('Failed to initialize in-memory Chroma vector store')
    raise RuntimeError('Unable to initialize Chroma vector store without persistence.') from exc

# 父文档仓：chunk → parent_id → 全文
docstore = InMemoryStore()
# 本地聊天模型
model = ChatOllama(model=CHAT_MODEL)

# 通用切块：1000/150，按段落→行→空格退化
DEFAULT_TEXT_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=['\n\n', '\n', ' ', '']
)
# Markdown 先按标题切开，再交给递归切块
MARKDOWN_HEADERS = [('#', 'Header 1'), ('##', 'Header 2'), ('###', 'Header 3')]
MARKDOWN_SPLITTER = MarkdownHeaderTextSplitter(headers_to_split_on=MARKDOWN_HEADERS, strip_headers=False)


In [ ]:
# ========== 文件名清洗 / 扩展名推断 / 缓存路径 / 哈希 / manifest 条目 ==========

# 把任意文件名收成磁盘安全的一小段
def safe_filename(name: str, max_length: int = 120) -> str:
    sanitized = re.sub(r'[^A-Za-z0-9._-]', '_', name)
    sanitized = sanitized.strip('._') or 'untitled'
    return sanitized[:max_length]

# 优先用原名后缀，否则按 MIME / Google 导出规则
def determine_extension(metadata: dict) -> str:
    mime_type = metadata.get('mimeType', '')
    name = metadata.get('name')
    if name and Path(name).suffix:
        return Path(name).suffix.lower()
    if mime_type in GOOGLE_EXPORT_FORMATS:
        return GOOGLE_EXPORT_FORMATS[mime_type][1]
    return MIME_TYPE_TO_EXTENSION.get(mime_type, '.txt')

# 下载落盘路径：stem_fileId.ext，避免重名覆盖
def cached_file_path(metadata: dict) -> Path:
    file_id = metadata.get('id', 'unknown')
    extension = determine_extension(metadata)
    safe_name = safe_filename(Path(metadata.get('name', file_id)).stem)
    return DOWNLOAD_DIR / f'{safe_name}_{file_id}{extension}'

# 分块读文件算 SHA1，用于本地上传去重/版本
def hash_file(path: Path) -> str:
    digest = hashlib.sha1()
    with path.open('rb') as fh:
        while True:
            block = fh.read(HASH_BLOCK_SIZE)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

# 从 manifest 条目取出 modified 版本戳
def manifest_version(entry: dict | str | None) -> Optional[str]:
    if entry is None:
        return None
    if isinstance(entry, str):
        return entry
    if isinstance(entry, dict):
        return entry.get('modified')
    return None

# 写入/更新单条 manifest 记录
def update_manifest_entry(manifest: dict, *, file_id: str, modified: str, path: Path, mime_type: str, name: str) -> None:
    manifest[file_id] = {
        'modified': modified,
        'path': str(path),
        'mimeType': mime_type,
        'name': name,
        'file_type': Path(path).suffix.lower(),
    }


In [ ]:
# ========== Drive 列表与下载（含 Google Docs 导出）==========

# 按 MIME / 可选 folder_id 列出文件，支持 pageToken 翻页
def list_drive_text_files(service, folder_id: Optional[str], allowed_mime_types: list[str], limit: Optional[int]) -> list[dict]:
    # Drive 查询语言：未进垃圾桶 + MIME 过滤 + 可选父目录
    query_parts = ["trashed = false"]
    mime_types = allowed_mime_types or list(MIME_TYPE_TO_EXTENSION.keys())
    mime_clause = ' or '.join([f"mimeType = '{mime}'" for mime in mime_types])
    query_parts.append(f'({mime_clause})')
    if folder_id:
        query_parts.append(f"'{folder_id}' in parents")
    query = ' and '.join(query_parts)

    files: list[dict] = []
    page_token: Optional[str] = None

    # 翻页直到没有 nextPageToken 或达到 limit
    while True:
        page_size = min(100, limit - len(files)) if limit else 100
        if page_size <= 0:
            break
        try:
            response = service.files().list(
                q=query,
                spaces='drive',
                fields='nextPageToken, files(id, name, mimeType, modifiedTime)',
                orderBy='modifiedTime desc',
                pageToken=page_token,
                pageSize=page_size,
            ).execute()
        except HttpError as exc:
            raise RuntimeError(f'Google Drive API error: {exc}') from exc

        batch = response.get('files', [])
        files.extend(batch)
        if limit and len(files) >= limit:
            return files[:limit]
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    return files

# 下载或导出到本地缓存，并更新 manifest
def download_drive_file(service, metadata: dict, manifest: dict) -> Path:
    file_id = metadata['id']
    mime_type = metadata.get('mimeType', '')
    cache_path = cached_file_path(metadata)
    export_mime = None
    # Google Docs 等云端类型：走 export_media
    if mime_type in GOOGLE_EXPORT_FORMATS:
        export_mime, extension = GOOGLE_EXPORT_FORMATS[mime_type]
        if cache_path.suffix.lower() != extension:
            cache_path = cache_path.with_suffix(extension)


    # 普通二进制文件 get_media；云端文档 export
    request = (
        service.files().export_media(fileId=file_id, mimeType=export_mime)
        if export_mime
        else service.files().get_media(fileId=file_id)
    )

    logger.debug('Downloading %s (%s) -> %s', metadata.get('name', file_id), file_id, cache_path)
    with cache_path.open('wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        # MediaIoBaseDownload：分块拉完
        while not done:
            status, done = downloader.next_chunk()
            if status:
                logger.debug('Download progress %.0f%%', status.progress() * 100)

    update_manifest_entry(
        manifest,
        file_id=file_id,
        modified=metadata.get('modifiedTime', ''),
        path=cache_path,
        mime_type=mime_type,
        name=metadata.get('name', cache_path.name),
    )
    return cache_path


In [ ]:
# ========== 读文档：docx/txt/md → Document → 切块（parent/child id）==========

# 用 python-docx 抽段落文本
def extract_docx_text(path: Path) -> str:
    doc = DocxDocument(str(path))
    lines = [paragraph.text.strip() for paragraph in doc.paragraphs if paragraph.text.strip()]
    return '\n'.join(lines)

# 按后缀选择加载器，并写入统一 metadata
def load_documents(
    path: Path,
    *,
    source_id: Optional[str] = None,
    file_type: Optional[str] = None,
    modified: Optional[str] = None,
    display_name: Optional[str] = None,
 ) -> list[Document]:
    suffix = (file_type or path.suffix or '.txt').lower()
    try:
        if suffix in {'.txt', '.md'}:
            loader = TextLoader(str(path), encoding='utf-8')
            documents = loader.load()
        elif suffix == '.docx':
            documents = [Document(page_content=extract_docx_text(path), metadata={'source': str(path)})]
        else:
            raise ValueError(f'Unsupported file type: {suffix}')
    except UnicodeDecodeError as exc:
        raise ValueError(f'Failed to read {path}: {exc}') from exc

    base_metadata = {
        metadata_key(Metadata.SOURCE): str(path),
        metadata_key(Metadata.FILE_TYPE): suffix,
        metadata_key(Metadata.TITLE): display_name or path.name,
    }
    if source_id:
        base_metadata[metadata_key(Metadata.ID)] = source_id
    if modified:
        base_metadata[metadata_key(Metadata.MODIFIED)] = modified

    cleaned: list[Document] = []
    for doc in documents:
        content = doc.page_content.strip()
        if not content:
            continue
        merged_metadata = {**doc.metadata, **base_metadata}
        doc.page_content = content
        doc.metadata = merged_metadata
        cleaned.append(doc)
    return cleaned

# 丢掉空正文
def preprocess(documents: Iterable[Document]) -> list[Document]:
    return [doc for doc in documents if doc.page_content]

# Markdown 先按标题；再递归切块，并写 parent_id/chunk id
def chunk_documents(doc: Document) -> list[Document]:
    parent_id = doc.metadata.get(metadata_key(Metadata.ID))
    if not parent_id:
        raise ValueError('Document is missing a stable identifier for chunking.')

    # .md：先按标题切段，保留标题 metadata
    if doc.metadata.get(metadata_key(Metadata.FILE_TYPE)) == '.md':
        markdown_docs = MARKDOWN_SPLITTER.split_text(doc.page_content)
        seed_docs = [
            Document(page_content=section.page_content, metadata={**doc.metadata, **section.metadata})
            for section in markdown_docs
        ]
    else:
        seed_docs = [doc]

    # 统一再走递归字符切块
    chunks = DEFAULT_TEXT_SPLITTER.split_documents(seed_docs)
    for idx, chunk in enumerate(chunks):
        chunk.metadata[metadata_key(Metadata.PARENT_ID)] = parent_id
        chunk.metadata[metadata_key(Metadata.ID)] = f'{parent_id}::chunk-{idx:04d}'
        chunk.metadata.setdefault(metadata_key(Metadata.SOURCE), doc.metadata.get(metadata_key(Metadata.SOURCE)))
        chunk.metadata.setdefault(metadata_key(Metadata.TITLE), doc.metadata.get(metadata_key(Metadata.TITLE)))
    return chunks


In [ ]:
# ========== 同步主流程：授权 → 列表 → 增量下载/索引（Gradio 生成器）==========

# Gradio 生成器：yield (日志Markdown, sync_ready布尔)
def sync_drive_and_index(folder_id=None, selected_types=None, file_limit=None, _state: bool = False, progress=gr.Progress(track_tqdm=False)):
    # 空字符串视为「整个 My Drive」
    folder = (folder_id or '').strip() or None

    selections = selected_types if selected_types is not None else DEFAULT_FILE_TYPE_LABELS
    if not isinstance(selections, (list, tuple)):
        selections = [selections]
    selections = list(selections)

    # 至少选一种类型
    if len(selections) == 0:
        yield 'Select at least one file type before syncing.', False
        return

    chosen_keys: list[str] = []
    for item in selections:
        key = FILE_TYPE_LABEL_TO_KEY.get(item, item)
        if key in FILE_TYPE_OPTIONS:
            chosen_keys.append(key)

    if not chosen_keys:
        yield 'Select at least one file type before syncing.', False
        return

    # 把 UI 选项展开成 MIME 列表给 Drive 查询
    allowed_mime_types = sorted({mime for key in chosen_keys for mime in FILE_TYPE_OPTIONS[key]['mime_types']})

    limit: Optional[int] = None
    limit_warning: Optional[str] = None
    if file_limit not in (None, '', 0):
        try:
            parsed_limit = int(file_limit)
            if parsed_limit > 0:
                limit = parsed_limit
            else:
                raise ValueError
        except (TypeError, ValueError):
            limit_warning = 'File limit must be a positive integer. Syncing all matching files instead.'

    log_lines: list[str] = []

    # 累加日志行，方便前端一次渲染
    def push(message: str) -> str:
        log_lines.append(message)
        return '\n'.join(log_lines)

    if limit_warning:
        logger.warning(limit_warning)
        yield push(limit_warning), False

    # 第一步：OAuth
    progress(0, 'Authorizing Google Drive access...')
    yield push('Authorizing Google Drive access...'), False

    try:
        service = build_drive_service()
    except FileNotFoundError as exc:
        error_msg = f'Error: {exc}'
        logger.error(error_msg)
        yield push(error_msg), False
        return
    except Exception as exc:
        logger.exception('Drive authorization failed')
        error_msg = f'Error authenticating with Google Drive: {exc}'
        yield push(error_msg), False
        return

    list_message = 'Listing documents' + (f' (limit {limit})' if limit else '') + '...'
    progress(0, list_message)
    yield push(list_message), False

    try:
        files = list_drive_text_files(service, folder, allowed_mime_types, limit)
    except Exception as exc:
        logger.exception('Listing Drive files failed')
        error_msg = f'Error listing Google Drive files: {exc}'
        yield push(error_msg), False
        return

    total = len(files)
    if total == 0:
        info = 'No documents matching the selected types were found in Google Drive.'
        yield push(info), True
        return

    manifest = load_manifest()
    downloaded_count = 0

    for index, metadata in enumerate(files, start=1):
        file_id = metadata['id']
        name = metadata.get('name', file_id)
        remote_version = metadata.get('modifiedTime', '')
        manifest_entry = manifest.get(file_id)
        cache_path = cached_file_path(metadata)
        if isinstance(manifest_entry, dict) and manifest_entry.get('path'):
            cache_path = Path(manifest_entry['path'])
        cached_version = manifest_version(manifest_entry)

        if cached_version == remote_version and cache_path.exists():
            message = f"{index}/{total} Skipping cached file: {name} -> {cache_path}"
            progress(index / total, message)
            yield push(message), False
            continue

        download_message = f"{index}/{total} Downloading {name} -> {cache_path}"
        progress(max((index - 0.5) / total, 0), download_message)
        yield push(download_message), False

        try:
            downloaded_path = download_drive_file(service, metadata, manifest)
            index_message = f"{index}/{total} Indexing {downloaded_path.name}"
            progress(index / total, index_message)
            yield push(index_message), False
            index_document(
                downloaded_path,
                source_id=file_id,
                file_type=downloaded_path.suffix,
                modified=remote_version,
                display_name=name,
                manifest=manifest,
            )
            downloaded_count += 1
        except Exception as exc:
            error_message = f"{index}/{total} Failed to sync {name}: {exc}"
            logger.exception(error_message)
            progress(index / total, error_message)
            yield push(error_message), False

    if downloaded_count > 0:
        save_manifest(manifest)
        summary = f'Indexed {downloaded_count} new document(s) from Google Drive.'
    else:
        summary = 'Google Drive is already in sync.'

    progress(1, summary)
    yield push(summary), True


## RAG 流水线

把本地/Drive 文件读入 → 切块 → 写入向量库；`persist_vectorstore` 在本练习的内存模式下是空操作（会话结束不落盘）。


In [ ]:
# ========== 索引单文件：切块写入 Chroma + 父文档进 docstore ==========

def persist_vectorstore(_store) -> None:
    """内存模式占位：当前 Chroma 客户端不在会话间持久化。"""
    return


# 索引一个文件；若提供 manifest 且无 source_id，则按内容哈希登记
def index_document(
    file_path: Path | str,
    *,
    source_id: Optional[str] = None,
    file_type: Optional[str] = None,
    modified: Optional[str] = None,
    display_name: Optional[str] = None,
    manifest: Optional[dict] = None,
 ) -> tuple[str, int]:
    path = Path(file_path)
    # 归一成绝对路径
    path = path.expanduser().resolve()
    # Drive 用文件 id；本地上传用内容哈希当稳定 id
    resolved_id = source_id or f'local::{hash_file(path)}'
    documents = load_documents(
        path,
        source_id=resolved_id,
        file_type=file_type,
        modified=modified,
        display_name=display_name,
    )
    documents = preprocess(documents)
    if not documents:
        logger.warning('No readable content found in %s; skipping.', path)
        return resolved_id, 0

    total_chunks = 0
    for doc in documents:
        doc_id = doc.metadata.get(metadata_key(Metadata.ID), resolved_id)
        doc.metadata[metadata_key(Metadata.ID)] = doc_id
        # 先删旧 chunk，再写入，避免重复
        vectorstore.delete(where={metadata_key(Metadata.PARENT_ID): doc_id})
        chunks = chunk_documents(doc)
        if not chunks:
            continue
        vectorstore.add_documents(chunks)
        # 父文档存一份，检索后可预览全文
        docstore.mset([(doc_id, doc)])
        total_chunks += len(chunks)

    persist_vectorstore(vectorstore)
    if manifest is not None and not source_id:
        update_manifest_entry(
            manifest,
            file_id=resolved_id,
            modified=hash_file(path),
            path=path,
            mime_type=file_type or Path(path).suffix or '.txt',
            name=display_name or path.name,
        )
    return resolved_id, total_chunks


### 与 LLM 交互

检索带分数过滤 → 拼 context → 本地 ChatOllama 流式回答。prompt 要求**只依据 context**，避免胡编。


In [ ]:
# ========== 检索 + 拼 prompt + 流式 ask ==========

# 向量检索并按距离阈值过滤；可选打印 parent 预览
def retrieve_context(query: str, *, top_k: int = 8, distance_threshold: Optional[float] = SIMILARITY_DISTANCE_MAX):
    # 分数语义随后端距离函数而异；这里按「越大越远」阈值裁剪
    results_with_scores = vectorstore.similarity_search_with_score(query, k=top_k)
    logger.info(f'Matching records: {len(results_with_scores)}')

    filtered: list[tuple[Document, float]] = []
    for doc, score in results_with_scores:
        if score is None:
            continue
        score_value = float(score)
        print(f'DEBUG: Retrieved doc source={doc.metadata.get(metadata_key(Metadata.SOURCE))} distance={score_value}')
        # 超过阈值：视为不够相关，丢掉
        if distance_threshold is not None and score_value > distance_threshold:
            logger.debug(
                'Skipping %s with distance %.4f (above threshold %.4f)',
                doc.metadata.get(metadata_key(Metadata.SOURCE)),
                score_value,
                distance_threshold,
            )
            continue
        filtered.append((doc, score_value))

    if not filtered:
        return []

    for doc, score_value in filtered:
        parent_id = doc.metadata.get(metadata_key(Metadata.PARENT_ID))
        if parent_id:
            parent_doc = docstore.mget([parent_id])[0]
            if parent_doc and parent_doc.page_content:
                logger.debug(
                    'Parent preview (%s | %.3f): %s',
                    doc.metadata.get(metadata_key(Metadata.SOURCE), 'unknown'),
                    score_value,
                    parent_doc.page_content[:400].replace('\n', ' '),
                )

    return filtered


# 把命中文档格式化成带 Source/Distance 的 context 段落
def build_prompt_sections(relevant_docs: list[tuple[Document, float]]) -> str:
    sections: list[str] = []
    for idx, (doc, score) in enumerate(relevant_docs, start=1):
        source = doc.metadata.get(metadata_key(Metadata.SOURCE), 'unknown')
        snippet = doc.page_content.strip()[:MAX_CONTEXT_SNIPPET_CHARS]
        section = (
            f'[{idx}] Source: {source}\n'
            f'Distance: {score:.3f}\n'
            f'Content:\n{snippet}'
        )
        sections.append(section)
    return '\n\n'.join(sections)


# 流式问答：无命中则直接提示用户去同步/放宽过滤
def ask(message, history):
    relevant_docs = retrieve_context(message)
    if not relevant_docs:
        yield "I don't have enough information in the synced documents to answer that yet. Please sync additional files or adjust the filters."
        return

    # 英文 system 指令保留：约束模型只根据 context 回答
    context = build_prompt_sections(relevant_docs)
    prompt = f'''
    You are a retrieval-augmented assistant. Use ONLY the facts provided in the context to answer the user.
    If the context does not contain the answer, reply exactly: "I don't have enough information in the synced documents to answer that yet. Please sync additional files."
    
    Context:\n{context}
    '''

    messages = [
        ('system', prompt),
        ('user', message)
    ]

    # 本地流式生成
    stream = model.stream(messages)
    response_text = ''

    for chunk in stream:
        response_text += chunk.content or ''
        if not response_text:
            continue

        yield response_text


## Gradio 界面

左侧聊天（可上传文件即时索引），右侧 Google Drive 同步面板。`sync_state` 标记是否已成功同步。


In [ ]:
# ========== Gradio：ChatInterface + Drive Sync 面板 ==========

# 聊天入口：可先索引上传文件，再 ask；未同步且无上传则提示
def chat(message, history, sync_ready):
    if message is None:
        return ''

    # MultimodalTextbox：text + files
    text_input = message.get('text', '')
    files_uploaded = message.get('files', [])
    latest_file_path = Path(files_uploaded[-1]) if files_uploaded else None
    # 有上传：先 index_document，再决定是否继续问答
    if latest_file_path:
        manifest = load_manifest()
        doc_id, chunk_count = index_document(
            latest_file_path,
            file_type=latest_file_path.suffix,
            display_name=latest_file_path.name,
            manifest=manifest,
        )
        save_manifest(manifest)
        logger.info('Indexed upload %s as %s with %s chunk(s)', latest_file_path, doc_id, chunk_count)
        if not text_input:
            yield f'Indexed document from upload ({chunk_count} chunk(s)).'
            return

    if not text_input:
        return ''

    if not sync_ready and not files_uploaded:
        yield 'Sync Google Drive before chatting or upload a document first.'
        return

    for chunk in ask(text_input, history):
        yield chunk

# UI 标题（展示字符串保持英文）
title = "Drive Sage"
# fill_height + CUSTOM_CSS 让聊天列占满视口高度
with gr.Blocks(title=title, fill_height=True, css=CUSTOM_CSS) as ui:
    gr.Markdown(f'# {title}')
    gr.Markdown('## Search your Google Drive knowledge base with fully local processing.')
    # 会话内状态：是否已成功 sync
    sync_state = gr.State(False)

    with gr.Row():
        with gr.Column(scale=3, elem_id='chat-column'):
            gr.ChatInterface(
                fn=chat,
                chatbot=gr.Chatbot(height='80vh', elem_id='chat-output'),
                type='messages',
                textbox=gr.MultimodalTextbox(
                    file_types=['text', '.txt', '.md'],
                    autofocus=True,
                    elem_id='chat-input',
                ),
                additional_inputs=[sync_state],
            )
        with gr.Column(scale=2, min_width=320):
            gr.Markdown('### Google Drive Sync')
            drive_folder = gr.Textbox(
                label='Folder ID (optional)',
                placeholder='Leave blank to scan My Drive root',
            )
            file_types = gr.CheckboxGroup(
                label='File types to sync',
                choices=[config['label'] for config in FILE_TYPE_OPTIONS.values()],
                value=DEFAULT_FILE_TYPE_LABELS,
            )
            file_limit = gr.Number(
                label='Max files to sync (leave blank for all)',
                value=20,
            )
            sync_btn = gr.Button('Sync Google Drive')
            sync_status = gr.Markdown('No sync performed yet.')

            # 同步按钮：更新日志与 sync_state
            sync_btn.click(
                sync_drive_and_index,
                inputs=[drive_folder, file_types, file_limit, sync_state],
                outputs=[sync_status, sync_state],
            )

# debug=True：把报错打到笔记本更方便排查
ui.launch(debug=True)
